In [3]:
import hmac
import hashlib
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType

HMAC_KEY = b"UNA_CLAVE_SECRETA_LARGA_AQUI" # 32 bytes generada con CPRING

def hmac_sha256(value: str) -> str:
    if value is None:
        return None

    #Asegurar que value venga ya normalizado como string
    value_bytes = value.encode("utf-8")
    return hmac.new(HMAC_KEY, value_bytes, hashlib.sha256).hexdigest()

hmac_udf = udf(Hmac_sha256, StringType())

df_tokens = df.withColumn("rut_token", hmac_udf(col("rut_norm")))

In [ ]:
#Version con pandas
import pandas as pd

@pandas_udf("string")
def hmac_sha256_udf(col_series: pd.Series) -> pd.Series:
    def _one(v):
        if v is None:
            return None
        return hmac.new(HMAC_KEY, v.encode("utf-8"), hashlib.sha256).hexdigest()
    return col_series.apply(_one)

df_tokens = df.withColumn("rut_token", hmac_sha256_udf(col("rut_norm")))

Normalización previa:
- rut sin puntos, formato consistente: "12345678-9"
- cuentas sin espacios, ceros a la izquierda si aplica, etc

Gestión de la clave
- no dejar la clave harcodeada en el script en producción
- cargarla desde:
 - variable de entorno
 - secret manager

Determinismo
- No mezclar otros campos (fecha, id_transacción, etc) deltro del HMAC si se desea que el token sea estable por persona/cuenta

Algoritmo del dígito verificador

Para el número base del rut (sin DV, sin puntos) por ejemplo 12345678

-Tomar los digitos de derecha a izquierda
-multiplicarlos cíclicamente pode 2, 3, 4, 5, 6, 7 y repite
-sumar todos los productos
-calcular resto = suma % 11
- regla final
  - si dv_calc == 11 -> DV = '0'
  - si dv_calc == 10 -> DV = 'k'
  - en otro caso -> DV = str(dv_calc)

In [3]:
def rut_dv(numero: int | str) -> str:
    """
    Calcula el dígito verificador de un rut chileno.
    numero: parte numérica del rut, sin ountos ni DV (ej. 12345678)
    retorna: '0-9' o 'k'
    """
    num_str = str(numero)
    factores = [2, 3, 4, 5, 6, 7]

    suma = 0
    factor_idx = 0
    for d in reversed(num_str):
        suma += int(d) * factores[factor_idx]
        factor_idx = (factor_idx + 1) % len(factores)

    resto = suma % 11
    dv_calc = 11 - resto

    if dv_calc == 11:
        return '0'
    elif dv_calc == 10:
        return 'k'
    else:
        return str(dv_calc)

In [8]:
rut_dv(12345678)

'5'